In [ ]:
# Cell 1 — Run configuration
RUN_SEED = 1337  # CHANGE ONLY THIS: 42 / 1337 / 2026 / 31415 / 271828

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
if RUN_SEED not in PUBLICATION_SEEDS:
    raise ValueError(f"RUN_SEED must be one of {PUBLICATION_SEEDS}, got {RUN_SEED}")

MODEL_ID = 'OpenGVLab/InternVL3_5-8B-HF'
MODEL_REVISION = '3974c115d44f0e13eae8b654f680b1e561c71ae9'

# Validation-selected old Gated checkpoints verified from the five completed InternVL runs.
GATED_BEST_STEP_BY_SEED = {42: 932, 1337: 800, 2026: 600, 31415: 932, 271828: 932}

FACETGATE_COMMON_SHA256 = '735b3e08413949cdb148fe4ce570493b5cd410e562dad7a12748b2e5f2fd6a38'
PARENT_GATED_PROTOCOL_SHA256 = '627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd'
SHARED_FORWARD_PROTOCOL_SHA256 = '37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa'

print({
    "RUN_SEED": RUN_SEED,
    "expected_old_gated_best_step": GATED_BEST_STEP_BY_SEED[RUN_SEED],
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "TEST_forwards": 0,
})


In [ ]:
# Cell 2 — Same-seed Gated-parent discovery and cryptographic precheck
# Safe to run manually after CELL 1 before Save & Run All.
# No InternVL model load. No GPU training. No TEST forward.

import json, zipfile, hashlib, shutil, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RESOLVE_ROOT = WORK_ROOT / f"internvl_facegate_parent_seed{RUN_SEED}"
RESOLVE_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_STEP = int(GATED_BEST_STEP_BY_SEED[RUN_SEED])
EXPECTED_CP_NAMES = {f"checkpoint-{EXPECTED_STEP}", f"checkpoint-{EXPECTED_STEP:04d}"}

def _sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def _directory_sha256(path):
    """Exact Frozen-Common Gated checkpoint hash: all files except checkpoint_manifest.json."""
    path = Path(path)
    h = hashlib.sha256()
    files = sorted(
        p for p in path.rglob("*")
        if p.is_file() and p.name != "checkpoint_manifest.json"
    )
    for fp in files:
        rel = fp.relative_to(path).as_posix().encode("utf-8")
        h.update(rel + b"\0")
        h.update(_sha256_file(fp).encode("ascii") + b"\n")
    return h.hexdigest()

def _read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None

def _checkpoint_files_complete(cp):
    cp = Path(cp)
    required = [
        cp / "classifier_heads.pt",
        cp / "trainer_state.pt",
        cp / "rng_rank_0.pt",
        cp / "rng_rank_1.pt",
        cp / "adapter" / "adapter_config.json",
        cp / "checkpoint_manifest.json",
    ]
    if not all(p.is_file() for p in required):
        return False
    return (
        (cp / "adapter" / "adapter_model.safetensors").is_file()
        or (cp / "adapter" / "adapter_model.bin").is_file()
    )

def _root_for_checkpoint(cp):
    cp = Path(cp)
    if cp.parent.name == "checkpoints":
        return cp.parent.parent
    return cp.parent

def _validate_run_config(obj):
    if not isinstance(obj, dict):
        return False, "run_config_invalid"
    if int(obj.get("seed", -1)) != int(RUN_SEED):
        return False, f"run_config_seed={obj.get('seed')}"
    if obj.get("model_id") != MODEL_ID:
        return False, f"run_config_model_id={obj.get('model_id')}"
    if obj.get("model_revision") != MODEL_REVISION:
        return False, f"run_config_model_revision={obj.get('model_revision')}"
    if obj.get("frozen_common_gated_protocol_sha256") != PARENT_GATED_PROTOCOL_SHA256:
        return False, "run_config_gated_protocol_sha_mismatch"
    if obj.get("task_instruction_sha256") not in (
        None,
        "bfef1e522a9433e1ca0c57a3a1bea49cd890d8428308b7ffdf6b24e4fe51667f",
    ):
        return False, "run_config_task_instruction_sha_mismatch"
    if obj.get("allow_final_test") not in (None, False):
        return False, f"run_config_allow_final_test={obj.get('allow_final_test')}"
    return True, "ok"

def _validate_checkpoint_manifest(obj):
    if not isinstance(obj, dict):
        return False, "checkpoint_manifest_invalid"
    if int(obj.get("seed", -1)) != int(RUN_SEED):
        return False, f"checkpoint_seed={obj.get('seed')}"
    if int(obj.get("global_step", -1)) != EXPECTED_STEP:
        return False, f"checkpoint_step={obj.get('global_step')}"
    if obj.get("frozen_common_gated_protocol_sha256") != PARENT_GATED_PROTOCOL_SHA256:
        return False, "checkpoint_gated_protocol_sha_mismatch"
    if obj.get("common_scientific_hash") not in (None, PARENT_GATED_PROTOCOL_SHA256):
        return False, "checkpoint_common_scientific_hash_mismatch"
    fp = obj.get("checkpoint_content_sha256")
    if not (isinstance(fp, str) and len(fp) == 64 and all(c in "0123456789abcdef" for c in fp.lower())):
        return False, "checkpoint_content_sha256_missing_or_invalid"
    return True, "ok"

def _validate_thresholds(obj):
    if not isinstance(obj, dict):
        return False, "threshold_json_invalid"
    if obj.get("source_split") not in (None, "validation"):
        return False, f"threshold_source_split={obj.get('source_split')}"
    if int(obj.get("source_rows", 932)) != 932:
        return False, f"threshold_source_rows={obj.get('source_rows')}"
    th = obj.get("thresholds")
    if not isinstance(th, dict):
        return False, "threshold_mapping_missing"

    # IMPORTANT — exact InternVL Frozen-Common Gated threshold schema:
    #   4 binary thresholds:
    #       offensive, vulgar, abuse, sarcastic
    #   7 Target thresholds:
    #       target::Individual, target::Gender, target::Political,
    #       target::Social, target::National, target::Religious, target::Others
    #
    # Language is 3-class softmax/argmax and therefore has NO threshold.
    # Requiring a "language" threshold incorrectly rejects every valid
    # InternVL Gated thresholds.json.
    binary_required = {"offensive", "vulgar", "abuse", "sarcastic"}
    target_required = {
        "target::individual",
        "target::gender",
        "target::political",
        "target::social",
        "target::national",
        "target::religious",
        "target::others",
    }
    keys = {str(k).casefold() for k in th.keys()}

    missing_binary = sorted(binary_required - keys)
    missing_target = sorted(target_required - keys)

    if missing_binary:
        return False, f"binary_thresholds_missing={missing_binary}"
    if missing_target:
        return False, f"target_thresholds_missing={missing_target}"

    # No language threshold is expected or required.
    return True, "ok"

def _find_threshold_file(root):
    root = Path(root)
    candidates = [
        root / "artifacts" / "thresholds.json",
        root / "thresholds.json",
        root / "artifacts" / "validation_thresholds.json",
        root / "validation_thresholds.json",
    ]
    for p in candidates:
        if p.is_file():
            obj = _read_json(p)
            ok, _ = _validate_thresholds(obj)
            if ok:
                return p
    return None

def _direct_candidates():
    rows, rejected = [], []
    if not INPUT_ROOT.exists():
        return rows, rejected

    for mp in INPUT_ROOT.rglob("checkpoint_manifest.json"):
        cp = mp.parent
        if cp.name not in EXPECTED_CP_NAMES:
            continue

        if not _checkpoint_files_complete(cp):
            rejected.append({"source": str(cp), "reason": "checkpoint_files_incomplete"})
            continue

        root = _root_for_checkpoint(cp)
        rc_path = root / "run_config.json"
        if not rc_path.is_file():
            rejected.append({"source": str(cp), "reason": "run_config_missing"})
            continue
        rc = _read_json(rc_path)
        ok, reason = _validate_run_config(rc)
        if not ok:
            rejected.append({"source": str(cp), "reason": reason})
            continue

        mf = _read_json(mp)
        ok, reason = _validate_checkpoint_manifest(mf)
        if not ok:
            rejected.append({"source": str(cp), "reason": reason})
            continue

        th = _find_threshold_file(root)
        if th is None:
            rejected.append({"source": str(cp), "reason": "validation_thresholds_missing"})
            continue

        rows.append({
            "kind": "directory",
            "source": str(cp),
            "root": str(root),
            "thresholds": str(th),
            "run_config": str(rc_path),
            "manifest": str(mp),
            "fingerprint": str(mf["checkpoint_content_sha256"]),
        })

    return rows, rejected

def _zip_root_member(names, cp_prefix, basename):
    root_prefix = cp_prefix.split("checkpoints/")[0] if "checkpoints/" in cp_prefix else ""
    preferred = root_prefix + basename
    if preferred in names:
        return preferred
    hits = [n for n in names if n.endswith("/" + basename) or n == basename]
    if not hits:
        return None
    # Prefer the shortest item sharing the checkpoint's run prefix.
    hits = sorted(
        hits,
        key=lambda n: (
            0 if root_prefix and n.startswith(root_prefix) else 1,
            len(n),
            n,
        ),
    )
    return hits[0]

def _zip_candidates():
    rows, rejected = [], []
    if not INPUT_ROOT.exists():
        return rows, rejected

    for zp in INPUT_ROOT.rglob("*.zip"):
        try:
            # Skip obviously unrelated giant archives.
            if zp.stat().st_size > 8_000_000_000:
                continue
            with zipfile.ZipFile(zp) as zf:
                names = set(zf.namelist())
                manifests = [
                    n for n in names
                    if n.endswith("checkpoint_manifest.json")
                    and n.rstrip("/").split("/")[-2] in EXPECTED_CP_NAMES
                ]
                for mm in manifests:
                    cp_prefix = mm[:-len("checkpoint_manifest.json")]

                    required = {
                        cp_prefix + "classifier_heads.pt",
                        cp_prefix + "trainer_state.pt",
                        cp_prefix + "rng_rank_0.pt",
                        cp_prefix + "rng_rank_1.pt",
                        cp_prefix + "adapter/adapter_config.json",
                    }
                    adapter_ok = (
                        cp_prefix + "adapter/adapter_model.safetensors" in names
                        or cp_prefix + "adapter/adapter_model.bin" in names
                    )
                    if not required.issubset(names) or not adapter_ok:
                        continue

                    rc_member = _zip_root_member(names, cp_prefix, "run_config.json")
                    if rc_member is None:
                        continue
                    rc = json.loads(zf.read(rc_member).decode("utf-8"))
                    ok, reason = _validate_run_config(rc)
                    if not ok:
                        rejected.append({"source": str(zp), "reason": reason})
                        continue

                    mf = json.loads(zf.read(mm).decode("utf-8"))
                    ok, reason = _validate_checkpoint_manifest(mf)
                    if not ok:
                        rejected.append({"source": str(zp), "reason": reason})
                        continue

                    th_member = None
                    for basename in [
                        "artifacts/thresholds.json",
                        "thresholds.json",
                        "artifacts/validation_thresholds.json",
                        "validation_thresholds.json",
                    ]:
                        candidate = _zip_root_member(names, cp_prefix, basename)
                        if candidate is not None:
                            try:
                                th_obj = json.loads(zf.read(candidate).decode("utf-8"))
                                ok, _ = _validate_thresholds(th_obj)
                                if ok:
                                    th_member = candidate
                                    break
                            except Exception:
                                pass
                    if th_member is None:
                        continue

                    rows.append({
                        "kind": "zip",
                        "source": str(zp),
                        "checkpoint_prefix": cp_prefix,
                        "threshold_member": th_member,
                        "run_config_member": rc_member,
                        "manifest_member": mm,
                        "fingerprint": str(mf["checkpoint_content_sha256"]),
                    })
        except Exception:
            continue

    return rows, rejected

direct, rej1 = _direct_candidates()
zipped, rej2 = _zip_candidates()
discovered = direct + zipped
rejected = rej1 + rej2

# Same Gated checkpoint may appear once as a mounted directory and again
# inside Resume_Safeguard.zip. Scientific identity is the content SHA.
groups = {}
for r in discovered:
    groups.setdefault(r["fingerprint"], []).append(r)

parents = []
mirror_report = []
for fp, group in groups.items():
    group = sorted(
        group,
        key=lambda r: (
            0 if r.get("kind") == "directory" else 1,
            len(str(r.get("source", ""))),
            str(r.get("source", "")),
        ),
    )
    chosen = dict(group[0])
    chosen["mirror_count"] = len(group)
    chosen["mirror_sources"] = [
        {
            "kind": x.get("kind"),
            "source": x.get("source"),
            "checkpoint_prefix": x.get("checkpoint_prefix"),
        }
        for x in group
    ]
    parents.append(chosen)
    if len(group) > 1:
        mirror_report.append({
            "checkpoint_content_sha256": fp,
            "mirror_count": len(group),
            "canonical_kind": chosen["kind"],
            "canonical_source": chosen["source"],
        })

if len(parents) > 1:
    raise RuntimeError(
        "Multiple DIFFERENT valid same-seed InternVL Gated parents were found. "
        "Directory/ZIP mirrors with the same checkpoint_content_sha256 are allowed, "
        "but different hashes are a real ambiguity.\n"
        + json.dumps(parents, indent=2)
    )

PARENT = parents[0] if parents else None

# Canonicalize the chosen parent to a direct checkpoint directory and direct threshold file.
if PARENT is not None:
    expected_fp = PARENT["fingerprint"]

    if PARENT["kind"] == "directory":
        canonical_cp = Path(PARENT["source"])
        canonical_th = Path(PARENT["thresholds"])

    else:
        zp = Path(PARENT["source"])
        extract_root = RESOLVE_ROOT / ("zip_" + expected_fp[:16])
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zp) as zf:
            prefix = PARENT["checkpoint_prefix"]
            members = [n for n in zf.namelist() if n.startswith(prefix)]
            for n in members:
                rel = n[len(prefix):]
                if not rel or n.endswith("/"):
                    continue
                dest = extract_root / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(n) as fi, open(dest, "wb") as fo:
                    shutil.copyfileobj(fi, fo)

            canonical_th = RESOLVE_ROOT / ("thresholds_" + expected_fp[:16] + ".json")
            canonical_th.write_bytes(zf.read(PARENT["threshold_member"]))

        canonical_cp = extract_root

    if not _checkpoint_files_complete(canonical_cp):
        raise RuntimeError("Canonicalized old-Gated checkpoint is incomplete.")

    observed_fp = _directory_sha256(canonical_cp)
    if observed_fp != expected_fp:
        raise RuntimeError(
            "Old-Gated checkpoint content SHA mismatch after canonicalization: "
            f"manifest={expected_fp} observed={observed_fp}"
        )

    th_obj = _read_json(canonical_th)
    ok, reason = _validate_thresholds(th_obj)
    if not ok:
        raise RuntimeError(f"Canonical old-Gated threshold validation failed: {reason}")

    PARENT = dict(PARENT)
    PARENT["canonical_checkpoint"] = str(canonical_cp)
    PARENT["canonical_thresholds"] = str(canonical_th)
    PARENT["observed_checkpoint_content_sha256"] = observed_fp

EXECUTION_MODE = (
    "REUSE_EXISTING_GATED"
    if PARENT is not None
    else "SIMULTANEOUS_FROM_SCRATCH"
)

controller_record = {
    "schema": "M3Offend_9318_InternVL3_5_8B_HF448_FaceGate_AUTO_SEED_v1_0_controller",
    "seed": int(RUN_SEED),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "expected_old_gated_best_step": EXPECTED_STEP,
    "execution_mode": EXECUTION_MODE,
    "parent": PARENT,
    "duplicate_mirrors_collapsed": mirror_report,
    "rejected_candidates": rejected[:100],
    "facetgate_common_sha256": FACETGATE_COMMON_SHA256,
    "parent_gated_protocol_sha256": PARENT_GATED_PROTOCOL_SHA256,
    "shared_forward_protocol_sha256": SHARED_FORWARD_PROTOCOL_SHA256,
    "TEST_forwards": 0,
}
controller_path = WORK_ROOT / f"internvl_facegate_controller_seed{RUN_SEED}.json"
controller_path.write_text(json.dumps(controller_record, indent=2), encoding="utf-8")

print("=" * 100)
print("INTERNVL3.5 FACEGATE SAME-SEED GATED PRECHECK")
print("=" * 100)
print("RUN_SEED                 :", RUN_SEED)
print("Expected old Gated step  :", EXPECTED_STEP)
print("Physical valid locations :", len(discovered))
print("Unique scientific parents:", len(parents))
print("Execution mode           :", EXECUTION_MODE)
if PARENT is not None:
    print("Parent content SHA       :", PARENT["fingerprint"])
    print("Canonical checkpoint     :", PARENT["canonical_checkpoint"])
    print("Canonical thresholds     :", PARENT["canonical_thresholds"])
    if PARENT.get("mirror_count", 1) > 1:
        print("Directory/ZIP mirrors    :", PARENT["mirror_count"], "(collapsed)")
    print("NEXT: Gated is reused/frozen; FaceGate/SHTE Target only.")
else:
    print("NEXT: no same-seed Gated parent; full shared-forward Gated + FaceGate/SHTE.")
    if rejected:
        print("\nRejected Gated-looking candidates (diagnostic):")
        for rr in rejected[:20]:
            print(" -", rr.get("reason"), "::", rr.get("source"))
print("GPU/model loaded         : NO")
print("TEST model forwards      : 0")
print("=" * 100)


In [ ]:
# Cell 3 — Embedded verified execution paths
# Both children are embedded so the same notebook works in all accounts.
# The controller selects exactly one after the same-seed Gated precheck.

TARGET_ONLY_CHILD_ZLIB_B64 = '...'
SIMULTANEOUS_CHILD_ZLIB_B64 = '...'

TARGET_ONLY_CHILD_SHA256 = '15977a74decaf89c77f6c70b5e502a688f7d0b7ba7439836392a599a3b07b5c8'
SIMULTANEOUS_CHILD_SHA256 = 'b5ef0526654e27599664564245e01aff4872fae52689afa9643d2875f3cff17f'

print({
    "target_only_child_sha256": TARGET_ONLY_CHILD_SHA256,
    "simultaneous_child_sha256": SIMULTANEOUS_CHILD_SHA256,
    "TEST_forwards": 0,
})


In [ ]:
# Cell 4 — Materialize and statically audit the seed-specific execution branch
import base64, zlib, json, hashlib, re, ast
from pathlib import Path
import nbformat

def _decode_child(blob, expected_sha):
    raw = zlib.decompress(base64.b64decode(blob.encode("ascii")))
    observed = hashlib.sha256(raw).hexdigest()
    if observed != expected_sha:
        raise RuntimeError(
            f"Embedded child SHA mismatch: expected={expected_sha} observed={observed}"
        )
    return json.loads(raw.decode("utf-8"))

if EXECUTION_MODE == "REUSE_EXISTING_GATED":
    child = _decode_child(TARGET_ONLY_CHILD_ZLIB_B64, TARGET_ONLY_CHILD_SHA256)
else:
    child = _decode_child(SIMULTANEOUS_CHILD_ZLIB_B64, SIMULTANEOUS_CHILD_SHA256)

for cell in child.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    src = src.replace("__RUN_SEED__", str(int(RUN_SEED)))

    if EXECUTION_MODE == "REUSE_EXISTING_GATED":
        src = src.replace(
            '"__GATED_SOURCE__"',
            repr(str(PARENT["canonical_checkpoint"])),
        )
        src = src.replace(
            '"__GATED_THRESHOLDS__"',
            repr(str(PARENT["canonical_thresholds"])),
        )

    cell["source"] = src.splitlines(keepends=True)
    cell["execution_count"] = None
    cell["outputs"] = []

# Resolve cell IDs now so Kaggle/nbconvert does not emit MissingIDFieldWarning.
child_nb = nbformat.from_dict(child)
for i, cell in enumerate(child_nb.cells):
    if not getattr(cell, "id", None):
        cell.id = f"ivfg-{i:02d}-{hashlib.sha1(('seed'+str(RUN_SEED)+str(i)).encode()).hexdigest()[:8]}"
nbformat.validate(child_nb)
child = json.loads(nbformat.writes(child_nb))

all_code = "\n".join(
    "".join(c.get("source", []))
    for c in child.get("cells", [])
    if c.get("cell_type") == "code"
)

# No seed placeholder may survive.
if "__RUN_SEED__" in all_code:
    raise RuntimeError("Generated child still contains unresolved __RUN_SEED__.")

if EXECUTION_MODE == "REUSE_EXISTING_GATED":
    if "__GATED_SOURCE__" in all_code or "__GATED_THRESHOLDS__" in all_code:
        raise RuntimeError("Generated target-only child still contains unresolved parent placeholders.")
    if str(PARENT["canonical_checkpoint"]) not in all_code:
        raise RuntimeError("Generated target-only child is missing the exact controller-bound checkpoint.")
    if str(PARENT["canonical_thresholds"]) not in all_code:
        raise RuntimeError("Generated target-only child is missing the exact controller-bound thresholds.")
    required_tokens = [
        FACETGATE_COMMON_SHA256,
        PARENT_GATED_PROTOCOL_SHA256,
        "class FacetGate",
        "PARENT GATED CHECKPOINT",
        "TEST",
    ]
else:
    required_tokens = [
        FACETGATE_COMMON_SHA256,
        PARENT_GATED_PROTOCOL_SHA256,
        SHARED_FORWARD_PROTOCOL_SHA256,
        "SIMULTANEOUS_FROM_SCRATCH",
        "SHARED-FORWARD FACETGATE/SHTE UPDATE",
        "facetgate_compatibility_replay",
        "fixed_gated_validation",
    ]

missing = [t for t in required_tokens if t not in all_code]
if missing:
    raise RuntimeError(f"Generated child is missing required scientific/runtime invariants: {missing}")

# Compile every generated notebook code cell.
for i, c in enumerate(child.get("cells", [])):
    if c.get("cell_type") == "code":
        compile("".join(c.get("source", [])), f"generated_child_cell_{i}", "exec")

# Extra: in the simultaneous path, extract the literal worker source from CELL 4
# and compile the actual torchrun worker itself, not only the notebook wrapper.
if EXECUTION_MODE == "SIMULTANEOUS_FROM_SCRATCH":
    worker_sources = []
    for c in child.get("cells", []):
        if c.get("cell_type") != "code":
            continue
        src = "".join(c.get("source", []))
        if "WORKER_SOURCE =" not in src:
            continue
        tree = ast.parse(src)
        for node in tree.body:
            if isinstance(node, ast.Assign):
                if any(isinstance(t, ast.Name) and t.id == "WORKER_SOURCE" for t in node.targets):
                    worker_sources.append(ast.literal_eval(node.value))
    if len(worker_sources) != 1:
        raise RuntimeError(f"Expected exactly one embedded InternVL worker source, found {len(worker_sources)}")
    worker_source = worker_sources[0]
    compile(worker_source, "internvl_facegate_shared_forward_worker.py", "exec")
    worker_sha = hashlib.sha256(worker_source.encode("utf-8")).hexdigest()

    required_worker_tokens = [
        "SHARED-FORWARD FACETGATE/SHTE UPDATE",
        "FACETGATE_COMMON_SHA256",
        "SHARED_FORWARD_PROTOCOL_SHA256",
        "save_facet_candidate",
        "facetgate_compatibility_replay",
        "test_model_forwards",
    ]
    miss_worker = [x for x in required_worker_tokens if x not in worker_source]
    if miss_worker:
        raise RuntimeError(f"InternVL shared-forward worker invariant loss: {miss_worker}")
else:
    worker_sha = None

child_name = (
    f"M3Offend_9318_InternVL3_5_8B_HF448_FaceGate_Seed{RUN_SEED}_TARGET_ONLY.ipynb"
    if EXECUTION_MODE == "REUSE_EXISTING_GATED"
    else f"M3Offend_9318_InternVL3_5_8B_HF448_FaceGate_Seed{RUN_SEED}_FULL.ipynb"
)
CHILD_NOTEBOOK = WORK_ROOT / child_name
CHILD_NOTEBOOK.write_text(json.dumps(child, ensure_ascii=False, indent=1), encoding="utf-8")
CHILD_SHA256 = hashlib.sha256(CHILD_NOTEBOOK.read_bytes()).hexdigest()

print("[CHILD STATIC + SEMANTIC QA PASS]", {
    "seed": RUN_SEED,
    "execution_mode": EXECUTION_MODE,
    "child_notebook": str(CHILD_NOTEBOOK),
    "child_sha256": CHILD_SHA256,
    "embedded_worker_sha256": worker_sha,
    "TEST_forwards": 0,
})


In [ ]:
# CELL 5 — EXECUTE THE AUTOMATICALLY SELECTED PIPELINE IN A FRESH KERNEL
# The child owns the package/runtime/GPU lifecycle; this controller remains lightweight.

import subprocess, sys, time
from pathlib import Path

EXECUTED_NOTEBOOK = CHILD_NOTEBOOK.with_name(
    CHILD_NOTEBOOK.stem + "_EXECUTED.ipynb"
)

cmd = [
    sys.executable,
    "-m",
    "jupyter",
    "nbconvert",
    "--to",
    "notebook",
    "--execute",
    "--ExecutePreprocessor.timeout=-1",
    "--output",
    EXECUTED_NOTEBOOK.name,
    str(CHILD_NOTEBOOK),
]

print("[EXECUTING]", " ".join(cmd), flush=True)
started = time.time()
proc = subprocess.run(cmd, cwd=str(WORK_ROOT))
elapsed = time.time() - started

if proc.returncode != 0:
    raise RuntimeError(
        f"Selected InternVL3.5 FaceGate pipeline failed: "
        f"mode={EXECUTION_MODE} returncode={proc.returncode}. "
        f"Inspect {EXECUTED_NOTEBOOK if EXECUTED_NOTEBOOK.exists() else CHILD_NOTEBOOK}"
    )

print("[PIPELINE PROCESS EXITED CLEANLY]", {
    "seed": RUN_SEED,
    "execution_mode": EXECUTION_MODE,
    "elapsed_seconds": elapsed,
    "executed_notebook": str(EXECUTED_NOTEBOOK),
    "TEST_forwards": 0,
})


In [ ]:
# CELL 6 — FINAL CONTROLLER SUMMARY
# This does not invent a success state: it reports what artifacts actually exist.

import json
from pathlib import Path

print("=" * 100)
print("M3Offend InternVL3.5 FaceGate AUTO-SEED STATUS")
print("=" * 100)
print("Seed :", RUN_SEED)
print("Mode :", EXECUTION_MODE)
print("Executed child :", EXECUTED_NOTEBOOK)

if EXECUTION_MODE == "REUSE_EXISTING_GATED":
    print("Route : exact same-seed old Gated reused/frozen -> frozen FacetGate v1.0 Target-only")
    print("Parent checkpoint :", PARENT["canonical_checkpoint"])
    print("Parent content SHA :", PARENT["fingerprint"])
    print("Parent thresholds  :", PARENT["canonical_thresholds"])
else:
    print("Route : full Shared-Forward Gated + detached FaceGate/SHTE from scratch")

# Find likely final status/result manifests created by either execution path.
candidates = []
for p in Path("/kaggle/working").rglob("final_status.json"):
    s = str(p).casefold()
    if "internvl" in s and ("facetgate" in s or "facegate" in s):
        candidates.append(p)

if candidates:
    print("\nFinal-status artifacts:")
    for p in sorted(candidates):
        print(" -", p)
        try:
            print(json.dumps(json.loads(p.read_text(encoding="utf-8")), indent=2)[:5000])
        except Exception:
            pass
else:
    print("\nNo final_status.json found yet; inspect the executed child and live logs.")

manifests = []
for name in [
    "dual_checkpoint_final_manifest.json",
    "facetgate_result.json",
    "run_manifest.json",
]:
    for p in Path("/kaggle/working").rglob(name):
        s = str(p).casefold()
        if "internvl" in s and ("facetgate" in s or "facegate" in s):
            manifests.append(p)

if manifests:
    print("\nResult manifests:")
    for p in sorted(set(manifests)):
        print(" -", p)

print("\nLocked TEST forwards expected in development : 0")
print("=" * 100)
